# 39 - Humanoid VLA Architecture

## What / Why / How

**What we are trying to do:** Understand VLA-style humanoid autonomy: perception, language, planning, action chunks, and low-level controllers.

**Why this matters:** Systems like GR00T, Gemini Robotics, OpenPI, SmolVLA, Xiaomi-Robotics-0, and Dexora point toward layered language-conditioned robot policies.

**How we will do it:** Build a tiny hierarchical policy mockup that turns a language command into skill selection and action chunks.

## Humanoid VLA Architecture

This notebook is a mock architecture, not a real foundation model. It teaches the interfaces between language, perception, skill selection, and low-level action.

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
skills = {
    "pick": {"preconditions": ["object_visible", "hand_free"], "action_dim": 7},
    "place": {"preconditions": ["holding_object", "target_visible"], "action_dim": 7},
    "walk": {"preconditions": ["path_clear"], "action_dim": 3},
    "look": {"preconditions": [], "action_dim": 2},
}

def parse_instruction(text):
    text = text.lower()
    if "pick" in text or "grab" in text:
        return "pick"
    if "place" in text or "put" in text:
        return "place"
    if "walk" in text or "go" in text:
        return "walk"
    return "look"

instruction = "pick up the red cup"
selected = parse_instruction(instruction)
print("selected skill:", selected, skills[selected])

In [ ]:
world_state = {"object_visible", "hand_free", "target_visible", "path_clear"}

def preconditions_ok(skill, state):
    missing = [p for p in skills[skill]["preconditions"] if p not in state]
    return missing

missing = preconditions_ok(selected, world_state)
print("missing preconditions:", missing)

In [ ]:
rng = np.random.default_rng(39)

def action_chunk_for_skill(skill, horizon=6):
    dim = skills[skill]["action_dim"]
    base = rng.normal(0, 0.02, size=(horizon, dim))
    if skill == "pick":
        base[:, 2] -= np.linspace(0, 0.08, horizon)
        base[:, -1] = np.linspace(0.08, 0.0, horizon)  # gripper close
    elif skill == "place":
        base[:, 2] += np.linspace(0, 0.08, horizon)
        base[:, -1] = np.linspace(0.0, 0.08, horizon)
    elif skill == "walk":
        base[:, 0] = 0.2
    return base

chunk = action_chunk_for_skill(selected)
print(chunk)

In [ ]:
safety_limits = np.array([0.05] * chunk.shape[1])
safe_chunk = np.clip(chunk, -safety_limits, safety_limits)
print("clipped elements:", int(np.sum(np.abs(chunk - safe_chunk) > 1e-9)))

## Exercises

1. Add a recovery skill when preconditions are missing.
2. Add a high-level planner that sequences pick then place.
3. Explain why action chunks still need low-level safety filters.